# Level 2a — From cells to cell types

### CAJAL Neuromics 2026 · Project 15: Mapping the Spatial Cellular Architecture of the Brain

**~ ½ day · graded**

> *"We have a molecular parts-list of the developing cortex. Which of those cell types can we
> recognise, cell-by-cell, in the tissue itself?"*

### 📖 The reveal

Levels 0–1 kept the dataset anonymous. Here it is: this is the **developing human neocortex**,
from **Wang et al., *Nature* 2025** (*"Molecular and cellular dynamics of the developing human
neocortex"*, Kriegstein lab). The authors built a paired single-nucleus **multiome** atlas (RNA +
ATAC) spanning first trimester → adolescence in two areas — prefrontal cortex (**PFC**) and primary
visual cortex (**V1**) — and, on a subset, ran **MERFISH** spatial transcriptomics with a 300-gene
panel. Level 2 reproduces the heart of their spatial analysis (their Fig. 2). Over 2a + 2b you will
go from raw spatial cells → cell types → tissue architecture → cell–cell communication.

**This notebook (2a)** does the first step: assign a **cell-type identity** to each of the 404,030
MERFISH cells, and *earn the right to trust those labels* — because everything in 2b is built on them.

**Learning objectives — by the end you can:**
- Explain why a 300-gene spatial panel is annotated by **reference mapping** from a matched
  single-cell atlas, rather than by clustering it from scratch.
- Run **CellMapper** (a CCA joint embedding + kNN label transfer) to move cell-type labels from the
  multiome RNA reference onto the spatial cells.
- **Validate** a transferred annotation three ways: canonical markers, held-out-gene imputation, and
  agreement with an *independent* labelling.
- Judge annotation quality honestly — where it is trustworthy (lineage/class) and where it is not
  (fine subtypes).

**How to work through this notebook**
- 🔬 **TASK** cells are for you to complete. 💡 **HINT** points the way, ❓ **QUESTION** is for
  reflection, ⚠️ **CHECKPOINT** tells you what to expect so you know you're on track.
- **Kernel:** use **`Spatial Brain (SIF)`** (top-right in JupyterLab).
- Yellow **`FutureWarning`** boxes are normal — these libraries evolve fast; they are *not* errors.
- Some cells run for **a few minutes** on 404k cells (loading the reference, the CellMapper steps) —
  a slow cell is working, not hung.
- **Data — read/write map (you all run this in parallel, so this matters):** the cohort and the
  reference are **read-only** on the shared project filesystem; everything you *write* (the annotated
  object, figures) goes into **your own repo's `data/` and `figures/`**. Never write to the shared
  paths. The reference is ~3 GB — loading it takes a minute or two; that's normal.


---
## 0. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
from cellmapper import CellMapper

sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, frameon=False, figsize=(4, 4))
%matplotlib inline

# Shared, READ-ONLY project data
DATA = "/shared/projects/tp_2630_ubordeaux_neuromics_184418/projects/C15/data"
COHORT_PATH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells_student.h5ad"
REF_PATH = f"{DATA}/wang2025_multiome/processed/wang2025_multiome_rna.h5ad"
GROUND_TRUTH_PATH = f"{DATA}/wang2025_merfish/processed/wang2025_merfish_cells.h5ad"  # authors' labels — used ONLY for the final comparison

# YOUR outputs -> your repo's data/ (git-ignored, per-student)
from spatialbrain import FilePaths

OUT = FilePaths.dataset("wang2025_merfish").create()
print("writing outputs under:", OUT.processed)

---
## 1. The spatial cohort

The MERFISH cohort is **six sections** (3 PFC + 3 V1, spanning second trimester → infancy),
**404,030 cells × 300 genes**. `X` arrives pre-filled with a normalised matrix (we overwrite it from
raw below); `layers['counts']` holds the raw
counts; `obsm['spatial']` the physical (x, y) coordinate of every cell; and `.obs` carries the
experimental design (`Sample_ID`, `Region`, age) plus Vizgen QC. The authors' cell-type and niche
columns have been **removed** — that's your job to reconstruct.


In [ ]:
cohort = sc.read_h5ad(COHORT_PATH)
cohort

🔬 **TASK 1.1 — Get oriented.** Using `cohort.obs`, print (1) how many cells per `Sample_ID`, and
(2) the cross-tabulation of `Region` (PFC/V1) against age
(`Estimated_postconceptional_age_in_days`). How many samples, regions and ages are there?

💡 **HINT:** `.value_counts()` on a `.obs` column tallies cells per category; `pd.crosstab` cross-tabulates two `.obs` columns against each other.


In [ ]:
# 1. cells per sample
...

# 2. Region x age
...

⚠️ **CHECKPOINT.** Six `Sample_ID`s, two `Region`s (PFC, V1), three age groups. The section sizes
range from ~30k to ~110k cells.


---
## 2. Quality control — already done, but look anyway

These 404,030 cells are the authors' **post-QC** set. Their thresholds (from the paper's methods):
cell **volume > 10 µm³**, **25 ≤ nCount_Vizgen ≤ 2000** transcripts, **nFeature_Vizgen > 10** genes.
You won't re-filter, but always *look* at the QC distributions of data you're handed.


In [ ]:
sc.pl.violin(cohort, ["nCount_Vizgen", "nFeature_Vizgen", "volume"], jitter=0, multi_panel=True)

❓ **QUESTION.** The panel is 300 genes and counts are capped at 2000. Compared with a
whole-transcriptome scRNA-seq dataset (thousands of genes per cell), what does this tell you about
how much information you have per cell to *cluster* cell types de novo — and why reference mapping
might be the better move here?


---
## 3. Normalise, and look at the batch structure

Standard log-normalisation from raw counts (total-count normalise to 10,000, then `log1p`), then a
quick PCA/UMAP **on a subsample** (404k cells is slow to embed and we only want a look) coloured by
sample.


In [ ]:
cohort.X = cohort.layers["counts"].copy()
sc.pp.normalize_total(cohort, target_sum=1e4)
sc.pp.log1p(cohort)

# quick embedding on a 40k subsample, just to see batch structure
sub = sc.pp.subsample(cohort, n_obs=40000, copy=True, random_state=0)
sc.pp.pca(sub, n_comps=50)
sc.pp.neighbors(sub, n_neighbors=15)
sc.tl.umap(sub)
sc.pl.umap(sub, color=["Sample_ID", "Region"], wspace=0.4)

❓ **QUESTION.** Do cells group mostly by biology or mostly by `Sample_ID`? If sections separate in
the embedding, de-novo clustering would partly chase batch, not cell type. Reference mapping sidesteps
this: every cell is compared to the *same* reference in a shared space. That's the approach we take next.


---
## 4. The annotation reference

We annotate by transferring labels from the **multiome RNA modality** — the same nuclei, fully
sequenced (all genes), with the authors' cell-type labels. A useful data-wrangling note lives behind
this file: in the original MuData the cell-type labels sit on the **global** `.obs`, *not* on
`mdata['RNA'].obs` (which is empty). We've pre-extracted the RNA modality with the labels attached and
genes filtered to those expressed in ≥ 10 nuclei — so you just load an AnnData.


In [ ]:
ref = sc.read_h5ad(REF_PATH)  # ~3 GB; takes a minute
print(ref)
print("\ncell types in reference:", ref.obs["type"].cat.categories.size)
print("panel genes present in reference:", cohort.var_names.isin(ref.var_names).sum(), "/", cohort.n_vars)

The reference carries **34** fine cell types; the 300-gene panel resolves **29** of them in the
spatial data. All 300 panel genes are present in the reference — those shared genes are what the
mapping will build on.


---
## 5. Annotate by reference mapping (CellMapper)

**CellMapper** builds a **joint embedding** of query (spatial) and reference (RNA) cells with **fast
CCA** (on the shared 300 genes), finds each spatial cell's nearest reference neighbours, and transfers
the `type` label across that kNN graph. On CPU we use `pynndescent` for the neighbour search.

A point worth holding onto: the authors annotated their spatial cells **independently** of any
reference — by clustering and marker genes. We're using an *orthogonal* method (mapping from the
molecular atlas). So when we compare to their labels at the end, agreement is genuine cross-validation,
not circular reasoning.


🔬 **TASK 5.1 — Transfer the labels.** Create a `CellMapper(query=cohort, reference=ref)`, build the
fast-CCA joint embedding, then `map` the reference's `"type"` column onto the cohort.

💡 **HINT:** Check the [CellMapper docs](https://cellmapper.readthedocs.io/) for the three-step API —
construct `CellMapper(query, reference)`, build the joint space with its **fast-CCA** method, then
`.map(...)` to transfer an `obs` column across the kNN. Two things to get right: tell `.map` to use the
CCA embedding as its representation, and pass **`only_yx=True`** — otherwise CellMapper builds four kNN
graphs when you only need query→reference (much slower on 404k cells). Use `knn_method="pynndescent"`
(CPU). When it finishes, `cohort.obs` gains `type_pred` (the label) and `type_conf` (a confidence).


In [ ]:
cm = ...
...
cohort.obs["type_pred"].value_counts()

⚠️ **CHECKPOINT.** You should see a `type_pred` column with on the order of ~29 distinct types, with
excitatory-neuron and progenitor types among the most abundant (this is developing cortex).


---
## 6. Validation I — do canonical markers land on the right types?

The cleanest sanity check uses genes **actually measured** in the panel (not imputed — that would be
circular). A dotplot of lineage markers grouped by `type_pred` should show each marker lighting up in
the expected cell type.


In [ ]:
markers = {
    "RG/progenitor": ["VIM", "HES1", "HOPX", "TFAP2C", "CRYAB"],
    "IPC-EN": ["EOMES", "HES6"],
    "Excitatory": ["SATB2", "NEUROD1", "CUX2", "RORB", "BCL11B", "FEZF2"],
    "Inhibitory": ["GAD1", "GAD2", "DLX2", "SST", "VIP", "SP8"],
    "Astrocyte": ["AQP4", "GJA1", "SLC1A2"],
    "OPC/Oligo": ["OLIG1", "PDGFRA", "PLP1"],
    "Vascular": ["CLDN5", "FLT1"],
}
markers = {k: [g for g in v if g in cohort.var_names] for k, v in markers.items()}
sc.pl.dotplot(cohort, markers, groupby="type_pred", standard_scale="var", figsize=(14, 8))

🔬 **TASK 6.1 — Read the dotplot.** Name two predicted types whose marker pattern clearly matches
their expected identity (e.g. an excitatory type expressing `SATB2`, an inhibitory type expressing
`GAD1`/`GAD2`). Are there any types whose markers look ambiguous?


---
## 7. Validation II — held-out-gene imputation

A stricter test of the mapping: **hide** some panel genes from the joint embedding, then ask the
mapping to **predict** their expression from the reference, and correlate predicted vs measured. If the
neighbours are biologically right, held-out genes are recoverable. CellMapper masks genes via a
`is_train` column (`mask_var`), and we pass **`target_libsize`** so imputed values are scaled to each
spatial cell's depth (measured over the *training* genes) — the correct way to compare to measured counts.


In [ ]:
rng = np.random.default_rng(0)
is_train = np.ones(cohort.n_vars, bool)
is_train[rng.choice(cohort.n_vars, size=60, replace=False)] = False  # hold out 60 genes
cohort.var["is_train"] = is_train
held_genes = cohort.var_names[~is_train]

ref_panel = ref[:, cohort.var_names].copy()  # impute only panel genes (not all 33k)
ref_panel.var["is_train"] = is_train

cm2 = CellMapper(cohort.copy(), ref_panel)
cm2.compute_fast_cca(n_comps=50, mask_var="is_train")  # embedding uses training genes only
libsize = np.asarray(cohort[:, is_train].layers["counts"].sum(1)).ravel()
cm2.map(layer_key="X", use_rep="X_cca", n_neighbors=30, knn_method="pynndescent", target_libsize=libsize, only_yx=True)

imp = cm2.query_imputed[:, held_genes].X
imp = np.asarray(imp.todense()) if hasattr(imp, "todense") else np.asarray(imp)
true = cohort[:, held_genes].X
true = np.asarray(true.todense()) if hasattr(true, "todense") else np.asarray(true)
corr = np.array([np.corrcoef(true[:, j], imp[:, j])[0, 1] for j in range(len(held_genes))])
print(f"held-out genes: {len(held_genes)} | median Pearson r = {np.nanmedian(corr):.3f}")
plt.hist(corr, bins=20)
plt.xlabel("per-gene correlation (imputed vs measured)")
plt.ylabel("genes")
plt.show()

⚠️ **CHECKPOINT.** The median per-gene correlation is modest (around **0.3**) with a spread — some
genes are recovered well, others poorly. That is the honest reality of imputing from 240 genes: the
mapping captures broad lineage-level expression, not every gene's fine detail. Good enough to trust the
*labels*; a caution against over-reading *imputed* values.

❓ **QUESTION.** Why does using measured markers (§6) and held-out genes (§7) validate the annotation
*without* ever looking at the authors' labels — and why is that important for a fair test?


---
## 8. Validation III — an external opinion (CellTypist)

CellMapper used *this paper's own* reference. A useful cross-check is a **pretrained, external**
classifier that never saw this dataset. We run two CellTypist models and read them with a critical eye
— both have a **stage/region gap** with our second-trimester→infancy neocortex:
`Developing_Human_Brain` (first-trimester) and `Adult_Human_PrefrontalCortex` (adult).

🔬 **TASK 8.1 — Run the two baselines.** Download the two models, `celltypist.annotate` the cohort with
each (`majority_voting=False`), and store each model's predicted labels in a new `cohort.obs` column.
💡 **HINT:** `celltypist.annotate` returns a result object; its `.predicted_labels` holds the per-cell
calls. Loop over the two model names, annotate, and stash each result into its own `obs` column.
CellTypist expects log1p-CP10k input — exactly how we normalised the cohort. See the
[CellTypist docs](https://celltypist.readthedocs.io/).


In [ ]:
from celltypist import models

models.download_models(model=["Developing_Human_Brain.pkl", "Adult_Human_PrefrontalCortex.pkl"])
# annotate the cohort with each model and store predicted_labels in cohort.obs[f"ct_{model}"]
...

❓ **QUESTION.** The external models return labels like *"Hindbrain radial glia"* or *"Oligo MOG
OPALIN"* — plausible cell classes but from the wrong stage/region. What does this tell you about the
value of a **stage-matched, same-study reference** (our multiome) over a generic pretrained one?


---
## 9. The orthogonal comparison — how did we do?

Now we allow ourselves the authors' labels (in the original file), purely to *evaluate*. Remember
these were made **independently** (clustering + markers), so agreement with our reference-mapped labels
is a real cross-validation. We compare at two resolutions: fine **type** and coarse **class**.

⚠️ **A taxonomy caveat first.** The label spaces aren't identical: the multiome reference carries **34
types / 6 classes** (including an `Unknown` bucket and extra progenitor/immature types), while the
MERFISH ground truth resolves only **29 types / 5 classes**. So our mapper can emit a handful of labels
that simply *cannot* exist in the ground truth — those count as "disagreements" that are really
granularity mismatches, not annotation failures. We quantify how many, so the numbers aren't misread.


In [ ]:
truth = sc.read_h5ad(GROUND_TRUTH_PATH, backed="r")
cohort.obs["type_true"] = pd.Categorical(truth.obs.loc[cohort.obs_names, "type"].values)
cohort.obs["class_true"] = pd.Categorical(truth.obs.loc[cohort.obs_names, "class"].values)

from sklearn.metrics import accuracy_score

t2c = ref.obs.drop_duplicates("type").set_index("type")["class"].to_dict()
cohort.obs["class_pred"] = cohort.obs["type_pred"].map(t2c).astype(str)

gt_types = set(cohort.obs["type_true"].astype(str))
gt_classes = set(cohort.obs["class_true"].astype(str))
off = ~cohort.obs["type_pred"].astype(str).isin(gt_types)
print(f"predicted labels with no ground-truth counterpart: {off.sum()} cells ({100 * off.mean():.1f}%)")
print(
    "type-level agreement :",
    round(accuracy_score(cohort.obs["type_true"].astype(str), cohort.obs["type_pred"].astype(str)), 3),
)
print(
    "class-level agreement:", round(accuracy_score(cohort.obs["class_true"].astype(str), cohort.obs["class_pred"]), 3)
)
comparable = cohort.obs["class_pred"].isin(gt_classes)  # drop the rare 'Unknown' predictions
print(
    "class-level agreement (comparable cells only):",
    round(accuracy_score(cohort.obs["class_true"].astype(str)[comparable], cohort.obs["class_pred"][comparable]), 3),
)

# where do disagreements go? confusion at the class level (rows sum to 1)
cm_tab = pd.crosstab(cohort.obs["class_true"], cohort.obs["class_pred"], normalize="index")
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.heatmap(cm_tab, annot=True, fmt=".2f", cmap="Blues")
plt.title("class: authors (rows) vs ours (cols)")
plt.show()

⚠️ **CHECKPOINT.** About **10%** of cells get a label outside the ground-truth taxonomy — mostly finer
progenitor/immature types the reference resolves but the 300-gene panel and the authors' 29-type scheme
do not. **Class-level agreement ≈ 0.85–0.90**, but **type-level ≈ 0.5** — and that gap *is* the lesson:
two independent methods strongly agree on **lineage** (neuron? glia? progenitor?) but diverge on **fine
subtype** (which layer-specific excitatory neuron?), where a 300-gene panel and different pipelines
genuinely see things differently.

❓ **QUESTION.** Look at the class confusion heatmap: which lineages are most confidently recovered, and
which get mixed? Does that match where you'd expect a 300-gene panel to struggle (e.g. distinguishing
closely related progenitor vs immature-neuron states)?


---
🚀 **Going further (optional — open-ended, no single right answer).**
- **Which types are genuinely unresolvable on 300 genes?** Cluster the *reference* on the panel genes
  only and find the pairs of types whose centroids sit closest. Does the spatial mapper confuse those
  same pairs (compare `type_true` vs `type_pred`)? If so, that is an information limit of the panel, not
  a mapping bug — argue the distinction.
- **How sensitive is the transfer to `k`?** Re-run the mapping with `n_neighbors` in {10, 20, 30, 50}
  and compare the held-out-gene correlations (§7). Where is the sweet spot, and why?


---
## 10. Save the annotated cohort

Write the annotated object to **your** repo `data/` — Level 2b picks it up from there. We keep the 300
measured genes, the predicted labels, and the spatial coordinates.


In [ ]:
out_path = OUT.processed / "cohort_annotated.h5ad"
cohort.write_h5ad(out_path)
print("wrote", out_path, f"({out_path.stat().st_size / 1e6:.0f} MB)")
print("obs carried forward:", [c for c in cohort.obs.columns if c in ("type_pred", "type_conf", "Sample_ID", "Region")])

---
## Where this is heading

You've put a trustworthy cell-type label on every spatial cell — and you know *how far* to trust it
(lineage: yes; fine subtype: with caution). In **Level 2b** we stop asking *what* the cells are and
start asking *where* they are and *how they talk*: we'll place these types in physical space, discover
that they self-organise into the **cortical layers and zones** (niches), and trace the **cell–cell
communication** that helps build the cortex — reproducing the core of Wang et al.'s Figure 2.


In [ ]:
import session_info2

session_info2.session_info(os=True, cpu=True)